<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-09-multimodal-and-pretrained/lesson-9.2-gen-media/practice/GCP_Capstone_9.2_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 9.2 — Generative Media

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: authenticate, install SDKs, and init the Vertex client

Run this cell first. Everything below depends on `client`, `tts_client`, and the shared constants defined here.

In [ ]:
%%bash
pip install -q google-genai google-cloud-texttospeech Pillow

In [ ]:
# Application Default Credentials on Colab (no API keys)
from google.colab import auth
auth.authenticate_user()

from google import genai
from google.genai import types
from google.cloud import texttospeech
from PIL import Image
from io import BytesIO
import os

# --- Course constants ---
PROJECT_ID = 'documind-ai-YOUR-ID'   # replace with your project id
LOCATION   = 'us-central1'           # 'asia-south1' for India prod
USD_INR    = 85

# Model ids (swappable — re-verify GA status on the live model page)
TEXT_MODEL  = 'gemini-3.6-flash'       # default
IMAGE_MODEL = 'gemini-3-flash-image'   # Gemini native image generation

# Unified Gemini SDK on Vertex AI
client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)

# Cloud Text-to-Speech (Chirp 3 HD) — uses the same ADC credentials
tts_client = texttospeech.TextToSpeechClient()

print('SDKs ready — Vertex client + Chirp 3 HD TTS initialised')

## Exercise 1: Generate an infographic

**Difficulty:** Easy

Use Gemini native image generation to create a summary infographic from 3 data points. Save as PNG.

1. Prompt the image model with three data points and a title.
2. Request interleaved `TEXT` + `IMAGE` output.
3. Parse the response parts and save the image bytes to `infographic.png`.

**Expected behaviour:** PNG file with a labeled infographic.

In [ ]:
# Generate an infographic from data points (lesson Cell 1)
response = client.models.generate_content(
    model=IMAGE_MODEL,
    contents='Create a clean professional infographic with teal color scheme showing: '
             '1) 85% cost reduction 2) 3x faster processing 3) 99.2% accuracy. '
             'Include data labels and a title: DocuMind AI Results.',
    config=types.GenerateContentConfig(
        response_modalities=['TEXT', 'IMAGE'],
    ),
)

# Parse interleaved text + image response
for part in response.candidates[0].content.parts:
    if part.text:
        print('Text:', part.text)
    elif part.inline_data:
        img = Image.open(BytesIO(part.inline_data.data))
        img.save('infographic.png')
        print(f'Image saved: {img.size}')
        display(img)

## Exercise 2: Hindi TTS narration

**Difficulty:** Easy

Synthesize a Hindi summary with the Chirp 3 HD Kore voice. Save as MP3.

1. Build a `SynthesisInput` with your Hindi summary text.
2. Select the `hi-IN` language with the `hi-IN-Chirp3-HD-Kore` voice.
3. Request MP3 encoding and write the bytes to `narration_hi.mp3`.

**Expected behaviour:** MP3 file with natural Hindi narration.

In [ ]:
# Hindi narration with Chirp 3 HD Kore (adapted from lesson Cell 3)
response = tts_client.synthesize_speech(
    input=texttospeech.SynthesisInput(
        text='नमस्ते। यह आपके दस्तावेज़ का सारांश है। '
             'मुख्य निष्कर्ष: 85 प्रतिशत लागत में कमी और 3 गुना तेज़ प्रसंस्करण।'
    ),
    voice=texttospeech.VoiceSelectionParams(
        language_code='hi-IN',
        name='hi-IN-Chirp3-HD-Kore',
    ),
    audio_config=texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    ),
)

with open('narration_hi.mp3', 'wb') as f:
    f.write(response.audio_content)
print(f'Hindi narration: {len(response.audio_content)//1024} KB -> narration_hi.mp3')

## Exercise 3: Multi-language narration

**Difficulty:** Easy

Generate the same summary in Hindi, Telugu, and Tamil. Compare voice quality.

1. Map each Indian language code to a Chirp 3 HD voice and its summary text.
2. Loop over the map, synthesizing MP3 for each.
3. Save `narration_<lang>.mp3` for each and print the size.

**Expected behaviour:** 3 MP3 files in different languages.

In [ ]:
# Multi-language Indian narration (lesson Cell 4)
languages = {
    'hi-IN': ('hi-IN-Chirp3-HD-Kore',   'दस्तावेज़ का सारांश: 85% लागत में कमी।'),
    'te-IN': ('te-IN-Chirp3-HD-Charon',  'దస్తావేజు సారాంశం: 85% ఖర్చు తగ్గింది.'),
    'ta-IN': ('ta-IN-Chirp3-HD-Fenrir',  'ஆவண சுருக்கம்: 85% செலவு குறைப்பு.'),
}

for lang_code, (voice_name, text) in languages.items():
    response = tts_client.synthesize_speech(
        input=texttospeech.SynthesisInput(text=text),
        voice=texttospeech.VoiceSelectionParams(
            language_code=lang_code, name=voice_name),
        audio_config=texttospeech.AudioConfig(
            audio_encoding=texttospeech.AudioEncoding.MP3))
    filename = f'narration_{lang_code}.mp3'
    with open(filename, 'wb') as f:
        f.write(response.audio_content)
    print(f'{lang_code} ({voice_name}): {len(response.audio_content)//1024} KB -> {filename}')

## Exercise 4: Conversational image editing

**Difficulty:** Medium

Generate a chart, then refine it across multiple multi-turn chat messages.

1. Open a chat session on the image model with `TEXT` + `IMAGE` modalities.
2. Turn 1: generate a bar chart from four quarterly values.
3. Turn 2: refine — recolor the bars teal and add percentage labels.
4. Save the refined image and display each turn.

**Expected behaviour:** Progressive image refinement across turns.

In [ ]:
# Multi-turn image refinement (lesson Cell 2)
chat = client.chats.create(
    model=IMAGE_MODEL,
    config=types.GenerateContentConfig(
        response_modalities=['TEXT', 'IMAGE'],
    ),
)

# Turn 1: Generate
resp1 = chat.send_message('Create a bar chart showing Q1=45, Q2=62, Q3=78, Q4=91')
for part in resp1.candidates[0].content.parts:
    if part.inline_data:
        img1 = Image.open(BytesIO(part.inline_data.data))
        display(img1)

# Turn 2: Refine
resp2 = chat.send_message('Make the bars teal color and add percentage labels on top')
for part in resp2.candidates[0].content.parts:
    if part.inline_data:
        img2 = Image.open(BytesIO(part.inline_data.data))
        img2.save('chart_refined.png')
        display(img2)

print('Saved refined chart -> chart_refined.png')

## Exercise 5: Streaming TTS

**Difficulty:** Medium

Implement streaming synthesis for real-time audio delivery.

1. Build a `StreamingSynthesizeConfig` selecting a Chirp 3 HD streaming voice.
2. Yield the config request first, then a `StreamingSynthesisInput` per text chunk.
3. Iterate the streamed responses, appending each `audio_content` chunk as it arrives.
4. Save the accumulated PCM audio.

**Expected behaviour:** Audio chunks received in real-time.

In [ ]:
# Streaming synthesis — chunks arrive as they are generated
streaming_config = texttospeech.StreamingSynthesizeConfig(
    voice=texttospeech.VoiceSelectionParams(
        language_code='en-US', name='en-US-Chirp3-HD-Kore'))

text_chunks = [
    'Here is your document summary. ',
    'Cost reduction of 85 percent. ',
    'Processing is three times faster.',
]

def request_generator():
    # First request MUST carry only the config
    yield texttospeech.StreamingSynthesizeRequest(streaming_config=streaming_config)
    for chunk in text_chunks:
        yield texttospeech.StreamingSynthesizeRequest(
            input=texttospeech.StreamingSynthesisInput(text=chunk))

audio_buffer = bytearray()
for response in tts_client.streaming_synthesize(request_generator()):
    audio_buffer.extend(response.audio_content)
    print(f'Received chunk: {len(response.audio_content)} bytes (total {len(audio_buffer)})')

# Streaming output is raw PCM (LINEAR16)
with open('narration_stream.pcm', 'wb') as f:
    f.write(bytes(audio_buffer))
print(f'Streamed audio saved: {len(audio_buffer)//1024} KB -> narration_stream.pcm')

## Exercise 6: Voice cloning setup

**Difficulty:** Medium

Record a 10-second reference clip plus a consent clip, then call the voice-cloning key API.

1. Load the 10-second reference audio and the spoken-consent audio.
2. Provide the exact consent script the talent read aloud.
3. Call `generate_voice_cloning_key` to mint an instant custom-voice key.
4. Reuse the returned key in a `synthesize_speech` call via `VoiceCloneParams`.

**Expected behaviour:** A `voice_cloning_key` string is returned and usable for synthesis.

> Voice cloning requires verified consent audio. The exact request/field names track the Chirp 3 HD *instant custom voice* API — re-verify against the current `google-cloud-texttospeech` release.

In [ ]:
# Voice cloning key generation (Chirp 3 HD instant custom voice)
# You must supply your OWN reference + consent recordings.
#   reference_audio.wav : ~10s of the target voice speaking naturally
#   consent_audio.wav   : the talent reading the exact consent script below
CONSENT_SCRIPT = (
    'I am the owner of this voice and I consent to Google using this '
    'voice to create a synthetic voice model.'
)

try:
    with open('reference_audio.wav', 'rb') as f:
        reference_bytes = f.read()
    with open('consent_audio.wav', 'rb') as f:
        consent_bytes = f.read()

    key_response = tts_client.generate_voice_cloning_key(
        request=texttospeech.GenerateVoiceCloningKeyRequest(
            reference_audio=texttospeech.InputAudio(
                audio_config=texttospeech.AudioConfig(
                    audio_encoding=texttospeech.AudioEncoding.LINEAR16,
                    sample_rate_hertz=24000),
                content=reference_bytes),
            voice_talent_consent=texttospeech.InputAudio(
                audio_config=texttospeech.AudioConfig(
                    audio_encoding=texttospeech.AudioEncoding.LINEAR16,
                    sample_rate_hertz=24000),
                content=consent_bytes),
            consent_script=CONSENT_SCRIPT,
            language_code='en-US',
        )
    )
    voice_cloning_key = key_response.voice_cloning_key
    print('Voice cloning key generated:', voice_cloning_key[:24], '...')

    # Use the cloned voice to synthesize speech
    cloned = tts_client.synthesize_speech(
        input=texttospeech.SynthesisInput(text='This is my cloned voice reading the summary.'),
        voice=texttospeech.VoiceSelectionParams(
            language_code='en-US',
            voice_clone=texttospeech.VoiceCloneParams(
                voice_cloning_key=voice_cloning_key)),
        audio_config=texttospeech.AudioConfig(
            audio_encoding=texttospeech.AudioEncoding.MP3))
    with open('narration_cloned.mp3', 'wb') as f:
        f.write(cloned.audio_content)
    print(f'Cloned narration: {len(cloned.audio_content)//1024} KB -> narration_cloned.mp3')

except FileNotFoundError:
    print('Provide reference_audio.wav + consent_audio.wav to run voice cloning.')
except Exception as e:
    print(f'Voice cloning requires the instant-custom-voice API + consent: {e}')

## Exercise 7: Full generative pipeline

**Difficulty:** Challenge

Analyze a PDF, generate an infographic from the findings, then narrate it in 3 languages. Every output is auto-watermarked with SynthID.

1. Send a PDF to Gemini and extract the top data points.
2. Feed those points into the image model to build an infographic PNG.
3. Narrate the findings in Hindi, Telugu, and Tamil with Chirp 3 HD.
4. Note that SynthID watermarking is automatic on both image and audio output.

**Expected behaviour:** `infographic.png` + 3 MP3 files, all SynthID watermarked.

In [ ]:
# --- Step 1: Analyze a PDF with Gemini ---
# Upload your own PDF (e.g. via the Colab file browser) and set the path.
pdf_path = 'sample_report.pdf'

try:
    with open(pdf_path, 'rb') as f:
        pdf_bytes = f.read()

    analysis = client.models.generate_content(
        model=TEXT_MODEL,
        contents=[
            types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
            'Extract the 3 most important metrics from this document as short, '
            'punchy data points (one line each).',
        ],
    )
    findings = analysis.text
    print('Findings:\n', findings)
except FileNotFoundError:
    findings = '1) 85% cost reduction 2) 3x faster processing 3) 99.2% accuracy'
    print('No PDF found — using fallback findings:\n', findings)

In [ ]:
# --- Step 2: Generate an infographic from the findings (SynthID auto-watermarked) ---
img_resp = client.models.generate_content(
    model=IMAGE_MODEL,
    contents=f'Create a clean teal-themed infographic titled "DocuMind AI Results" '
             f'showing these data points with labels: {findings}',
    config=types.GenerateContentConfig(response_modalities=['TEXT', 'IMAGE']),
)
for part in img_resp.candidates[0].content.parts:
    if part.inline_data:
        Image.open(BytesIO(part.inline_data.data)).save('infographic.png')
        print('Saved infographic.png (SynthID pixel watermark embedded automatically)')

# --- Step 3: Narrate in 3 Indian languages (SynthID audio watermark auto) ---
pipeline_langs = {
    'hi-IN': 'hi-IN-Chirp3-HD-Kore',
    'te-IN': 'te-IN-Chirp3-HD-Charon',
    'ta-IN': 'ta-IN-Chirp3-HD-Fenrir',
}
for lang_code, voice_name in pipeline_langs.items():
    resp = tts_client.synthesize_speech(
        input=texttospeech.SynthesisInput(text=findings),
        voice=texttospeech.VoiceSelectionParams(language_code=lang_code, name=voice_name),
        audio_config=texttospeech.AudioConfig(audio_encoding=texttospeech.AudioEncoding.MP3))
    fn = f'pipeline_narration_{lang_code}.mp3'
    with open(fn, 'wb') as f:
        f.write(resp.audio_content)
    print(f'{lang_code}: {len(resp.audio_content)//1024} KB -> {fn}')

print('\nPipeline complete: infographic.png + 3 MP3s, all SynthID watermarked.')

## Exercise 8: SynthID text detection

**Difficulty:** Challenge

Use the `synthid-text` library with a `BayesianDetectorModel` to detect watermarks in text.

1. Install `synthid-text` and load a trained `BayesianDetectorModel`.
2. Configure the same watermarking keys used at generation time.
3. Score candidate text and threshold the Bayesian posterior.
4. Report `watermarked` / `not_watermarked` / `uncertain`.

**Expected behaviour:** Detection result: watermarked / not_watermarked / uncertain.

> SynthID for **images/audio** is automatic and detected via Google's portal — it is not an API you call. The `synthid-text` library is the programmatic path for **text** only, and needs a detector trained against your generation keys.

In [ ]:
%%bash
pip install -q synthid-text

In [ ]:
# Programmatic SynthID *text* detection with a Bayesian detector.
# Requires a BayesianDetectorModel trained on the same tournament keys
# that were used to watermark the text at generation time.
try:
    import torch
    from transformers import AutoTokenizer
    from synthid_text import detector_bayesian, logits_processing

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained('google/gemma-2b')

    # Watermarking config must match the generation-time keys
    watermark_config = logits_processing.SynthIDLogitsProcessor(
        keys=[654, 400, 836, 123, 340, 443, 597, 160, 57],
        ngram_len=5,
        device=DEVICE,
    )

    # Load a pre-trained detector (train your own on labelled samples in practice)
    detector = detector_bayesian.BayesianDetectorModel.from_pretrained(
        'google/synthid-text-detector').to(DEVICE)

    def classify(text, low=0.4, high=0.6):
        ids = tokenizer(text, return_tensors='pt').input_ids.to(DEVICE)
        g = watermark_config.compute_g_values(ids)
        mask = watermark_config.compute_mask(ids)
        score = detector(g, mask).item()
        if score >= high:
            return 'watermarked', score
        if score <= low:
            return 'not_watermarked', score
        return 'uncertain', score

    for sample in ['The quarterly report shows strong growth across all regions.',
                   'DocuMind reduced processing costs by 85 percent this quarter.']:
        label, score = classify(sample)
        print(f'[{label:16}] score={score:.3f} :: {sample[:50]}...')

except Exception as e:
    print('synthid-text detection needs a trained detector + matching keys.')
    print(f'Detail: {e}')
    print('Result vocabulary: watermarked / not_watermarked / uncertain')